# AWS Demo D. 에이전트의 외부 자격 증명 — AgentCore Identity 2LO · 3LO
**아주대학교 인공지능 보안 2회차 · AI 에이전트 보안 세션 보충 데모**

에이전트가 외부 API(GitHub, Google, 사내 API 등)를 부를 때 **비밀번호·API 키·토큰을 코드나 프롬프트에 넣지 않고** 받는 방법을 실습합니다.

| 구분 | 2LO (2-legged OAuth) | 3LO (3-legged OAuth) |
|---|---|---|
| OAuth 흐름 | Client Credentials | Authorization Code |
| AgentCore `oauth2Flow` | `M2M` | `USER_FEDERATION` |
| 누구의 권한? | **에이전트(앱) 자신** | **사용자 대신**(위임) — 사용자가 직접 로그인·동의 |
| 사람 개입 | 없음 | 최초 1회 브라우저 동의 |
| 토큰의 `sub` | 앱 클라이언트 ID | 사용자 ID |
| 예시 | 배치 리포트 에이전트가 사내 API 조회 | “내 캘린더/내 GitHub 저장소” 에이전트 |

| 단계 | 내용 | 소요 |
|---|---|---|
| 0 | 실습용 외부 IdP 만들기 (Amazon Cognito — Google·GitHub 대신) | 3분 |
| 1 | 워크로드 아이덴티티(에이전트의 신원) 만들기 | 1분 |
| 2 | **2LO**: M2M 토큰 발급 → 보호된 API 호출, SDK 데코레이터 | 4분 |
| 3 | **3LO**: 사용자 로그인·동의 → 사용자 위임 토큰, 토큰 볼트 재사용, 사용자 간 격리 | 6분 |
| 4 | 정리 | 1분 |

> 핵심 메시지: **에이전트는 비밀을 ‘들고 다니지’ 않습니다.** 자기 신원(워크로드 토큰)만 증명하고, 실제 외부 토큰은 AgentCore 토큰 볼트가 *필요한 순간에, 필요한 범위(scope)만큼, 해당 사용자 것만* 내어 줍니다.

### 준비
- 패키지: 바로 아래 설치 셀을 먼저 실행 (`boto3`, `bedrock-agentcore`, `pyjwt[crypto]`). 이미 boto3를 import한 커널이면 설치 후 **커널 재시작**
- 리전: `AWS_REGION` (기본 `us-west-2`, AgentCore 지원 리전)
- IAM 권한: `bedrock-agentcore:*WorkloadIdentity*`, `*Oauth2CredentialProvider*`, `GetWorkloadAccessToken*`, `GetResourceOauth2Token`, `CompleteResourceTokenAuth`, `CreateTokenVault`·`GetTokenVault`, `secretsmanager:CreateSecret·GetSecretValue·DeleteSecret·PutSecretValue`, `cognito-idp:*` (실습 풀 생성·삭제) — `iam-policy.json`의 `AgentCoreIdentityDemoD` 참고
- **3LO 콜백**: 노트북 커널이 도는 PC의 브라우저로 로그인하면 `localhost:9090` 콜백이 자동 처리됩니다. SageMaker 등 원격 커널이면 3-3의 **수동 완료 셀**을 쓰세요.
- 비용: Cognito(무료 티어 범위·M2M 토큰 소량), AgentCore Identity 호출 소량. 마지막 셀에서 모두 삭제합니다.

In [ ]:
# 패키지 설치 (SageMaker 등 처음 쓰는 커널에서 필수). 설치 후 아래 셀부터 실행
# bedrock-agentcore: 2-4·3-5 데코레이터 / pyjwt[crypto]: 2-3 서명 검증 (없으면 위조 토큰이 통과함)
%pip install -q --upgrade "boto3>=1.40" "bedrock-agentcore>=1.0" "pyjwt[crypto]>=2.8"

In [ ]:
import os, json, time, base64, secrets, threading, urllib.request
from urllib.parse import urlparse, parse_qs
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from IPython.display import display, HTML
from botocore.exceptions import ClientError

import aws_common as aws
from aws_common import show_table

os.environ.setdefault("AWS_REGION", aws.REGION)   # bedrock-agentcore SDK도 같은 리전을 쓰도록
ACCOUNT = aws.whoami()
REGION  = aws.REGION

cog  = aws.client("cognito-idp")
ctl  = aws.client("bedrock-agentcore-control")   # 컨트롤 플레인: 신원·자격 증명 공급자 생성
data = aws.client("bedrock-agentcore")           # 데이터 플레인: 토큰 발급

P            = f"{aws.PREFIX}-idd"                # 이 노트북 리소스 접두사
POOL_NAME    = f"{P}-idp"
API_ID       = f"{aws.PREFIX}-api"                # Cognito 리소스 서버(=보호된 API) 식별자
API_SCOPE    = f"{API_ID}/read"
WORKLOAD     = f"{P}-agent"
M2M_PROVIDER = f"{P}-m2m"
USR_PROVIDER = f"{P}-3lo"
RETURN_PORT  = 9090
RETURN_URL   = f"http://localhost:{RETURN_PORT}/oauth2/callback"
DEMO_USER, DEMO_PASS = "student01", "Ajou-Demo-" + secrets.token_hex(3) + "!"


def jwt_claims(token):
    """서명 검증 없이 JWT 페이로드만 보기 (관찰용)"""
    body = token.split(".")[1]
    return json.loads(base64.urlsafe_b64decode(body + "=" * (-len(body) % 4)))


def short(tok, n=24):
    return tok[:n] + "…" + f"({len(tok)}자)"

## 0. 실습용 외부 IdP — Amazon Cognito
실제로는 GitHub·Google·Okta·Entra ID 같은 **외부 OAuth 서버**가 이 자리에 옵니다. 여기서는 계정 하나로 끝내기 위해 Cognito 사용자 풀을 “외부 서비스의 OAuth 서버”로 씁니다.

- 리소스 서버 `ajou-ai-sec-api` + 스코프 `read` → 에이전트가 호출할 **보호된 API**
- 앱 클라이언트 2개: **M2M용**(client_credentials) / **사용자 위임용**(authorization code)
- 테스트 사용자 `student01`

In [ ]:
def cleanup_cognito():
    """같은 이름의 이전 실습 풀이 남아 있으면 도메인과 함께 삭제"""
    for p in cog.list_user_pools(MaxResults=60)["UserPools"]:
        if p["Name"] == POOL_NAME:
            dom = cog.describe_user_pool(UserPoolId=p["Id"])["UserPool"].get("Domain")
            if dom:
                cog.delete_user_pool_domain(Domain=dom, UserPoolId=p["Id"])
            cog.delete_user_pool(UserPoolId=p["Id"])
            print("[정리] 이전 사용자 풀 삭제", p["Id"])

cleanup_cognito()

POOL_ID = cog.create_user_pool(PoolName=POOL_NAME, AutoVerifiedAttributes=["email"])["UserPool"]["Id"]
DOMAIN  = f"{aws.PREFIX}-{ACCOUNT}-{secrets.token_hex(2)}"      # 전역 고유해야 함
cog.create_user_pool_domain(Domain=DOMAIN, UserPoolId=POOL_ID)

cog.create_resource_server(UserPoolId=POOL_ID, Identifier=API_ID, Name="Ajou demo API",
                           Scopes=[{"ScopeName": "read", "ScopeDescription": "read demo data"}])

# (A) 2LO용: 사람 없이 client_credentials
m2m_app = cog.create_user_pool_client(
    UserPoolId=POOL_ID, ClientName=f"{P}-m2m-client", GenerateSecret=True,
    AllowedOAuthFlowsUserPoolClient=True, AllowedOAuthFlows=["client_credentials"],
    AllowedOAuthScopes=[API_SCOPE])["UserPoolClient"]

# (B) 3LO용: 사용자 로그인 → authorization code. 콜백 URL은 1단계 이후 AgentCore 값으로 교체
USR_CLIENT_KW = dict(
    UserPoolId=POOL_ID, ClientName=f"{P}-user-client",
    AllowedOAuthFlowsUserPoolClient=True, AllowedOAuthFlows=["code"],
    AllowedOAuthScopes=["openid", "email", "profile", API_SCOPE],
    SupportedIdentityProviders=["COGNITO"],
    ExplicitAuthFlows=["ALLOW_REFRESH_TOKEN_AUTH"],
)
usr_app = cog.create_user_pool_client(GenerateSecret=True, CallbackURLs=["https://example.com/placeholder"],
                                      **USR_CLIENT_KW)["UserPoolClient"]
m2m_app.pop("ClientSecret"); usr_app.pop("ClientSecret")   # secret은 노트북 변수에 두지 않음

def client_secret(app):
    """AgentCore에 등록하는 순간에만 IdP에서 secret을 조회 (셀을 다시 실행해도 안전)"""
    return cog.describe_user_pool_client(UserPoolId=POOL_ID, ClientId=app["ClientId"])["UserPoolClient"]["ClientSecret"]

cog.admin_create_user(UserPoolId=POOL_ID, Username=DEMO_USER, MessageAction="SUPPRESS",
                      UserAttributes=[{"Name": "email", "Value": "student01@example.com"},
                                      {"Name": "email_verified", "Value": "true"}])
cog.admin_set_user_password(UserPoolId=POOL_ID, Username=DEMO_USER, Password=DEMO_PASS, Permanent=True)

# 도메인(호스티드 로그인 UI)이 ACTIVE 될 때까지 대기
for _ in range(60):
    st = cog.describe_user_pool_domain(Domain=DOMAIN)["DomainDescription"].get("Status")
    if st == "ACTIVE":
        break
    time.sleep(5)

ISSUER    = f"https://cognito-idp.{REGION}.amazonaws.com/{POOL_ID}"
HOSTED    = f"https://{DOMAIN}.auth.{REGION}.amazoncognito.com"
DISCOVERY = f"{ISSUER}/.well-known/openid-configuration"
print(f"[생성] 사용자 풀 {POOL_ID} · 도메인 {st} · {HOSTED}")
print(f"       테스트 사용자: {DEMO_USER} / {DEMO_PASS}")

In [ ]:
# 공통: AgentCore에 등록할 IdP 메타데이터 (OIDC discovery 문서 확인)
disc = json.load(urllib.request.urlopen(DISCOVERY, timeout=10))
print("authorization_endpoint:", disc.get("authorization_endpoint"))
print("token_endpoint        :", disc.get("token_endpoint"))

if disc.get("authorization_endpoint") and disc.get("token_endpoint"):
    OAUTH_DISCOVERY = {"discoveryUrl": DISCOVERY}
else:   # 도메인 전파 전이면 메타데이터를 직접 지정
    OAUTH_DISCOVERY = {"authorizationServerMetadata": {
        "issuer": ISSUER, "authorizationEndpoint": f"{HOSTED}/oauth2/authorize",
        "tokenEndpoint": f"{HOSTED}/oauth2/token", "responseTypes": ["code", "token"]}}

## 1. 워크로드 아이덴티티 — “에이전트의 신원”
AgentCore에서 에이전트는 **워크로드 아이덴티티**로 식별됩니다. AgentCore Runtime에 배포하면 자동으로 만들어지고, 요청마다 **워크로드 액세스 토큰**이 에이전트 코드에 주입됩니다(`BedrockAgentCoreContext`). 여기서는 직접 만들고 직접 발급받습니다.

`allowedResourceOauth2ReturnUrls` = 3LO 동의 후 사용자가 돌아올 수 있는 **허용 목록**. 공격자가 리다이렉트 주소를 바꿔 세션을 가로채는 것을 막습니다.

In [ ]:
def recreate(delete_fn, create_fn, **kw):
    try:
        delete_fn(name=kw["name"]); time.sleep(2)
    except ClientError as e:
        if e.response["Error"]["Code"] not in ("ResourceNotFoundException", "ValidationException"):
            raise
    return create_fn(**kw)

wi = recreate(ctl.delete_workload_identity, ctl.create_workload_identity,
              name=WORKLOAD, allowedResourceOauth2ReturnUrls=[RETURN_URL])
print("[생성] 워크로드 아이덴티티", wi["workloadIdentityArn"])
print("       허용된 return URL  :", wi["allowedResourceOauth2ReturnUrls"])

## 2. 2LO — 에이전트 자신의 권한 (M2M / client_credentials)
```
에이전트 ──(워크로드 토큰)──▶ AgentCore Identity ──(client_id+secret)──▶ IdP /oauth2/token
         ◀──────── access token (sub = 앱) ─────── 토큰 볼트 ◀───────────────┘
```
### 2-1. 자격 증명 공급자 등록
client secret은 **AgentCore가 Secrets Manager에 보관**합니다. 에이전트 코드·환경 변수·프롬프트 어디에도 secret이 없습니다.

In [ ]:
m2m_cp = recreate(ctl.delete_oauth2_credential_provider, ctl.create_oauth2_credential_provider,
    name=M2M_PROVIDER, credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput={"customOauth2ProviderConfig": {
        "oauthDiscovery": OAUTH_DISCOVERY,
        "clientId": m2m_app["ClientId"], "clientSecret": client_secret(m2m_app)}})

print("[생성] 자격 증명 공급자", m2m_cp["credentialProviderArn"])
print("       secret 보관 위치  :", m2m_cp["clientSecretArn"]["secretArn"])

### 2-2. 토큰 받기 (boto3 저수준 API)
1. `GetWorkloadAccessToken` — “나는 `ajou-ai-sec-idd-agent`다” (사용자 없음)
2. `GetResourceOauth2Token(oauth2Flow="M2M")` — 볼트에서 외부 API용 토큰 수령

In [ ]:
wat_m2m = data.get_workload_access_token(workloadName=WORKLOAD)["workloadAccessToken"]
print("워크로드 토큰:", short(wat_m2m))

def get_m2m_token():
    r = data.get_resource_oauth2_token(workloadIdentityToken=wat_m2m,
                                       resourceCredentialProviderName=M2M_PROVIDER,
                                       scopes=[API_SCOPE], oauth2Flow="M2M")
    return r["accessToken"]

tok_m2m = get_m2m_token()
c = jwt_claims(tok_m2m)
show_table([(k, c.get(k, "(없음)")) for k in ("sub", "client_id", "username", "scope", "token_use", "iss")],
           ["클레임", "값"])
print("\n→ sub == client_id, username 없음: '사용자'가 아니라 '앱'의 토큰입니다.")

In [ ]:
# 두 번째 호출: IdP에 다시 가지 않고 토큰 볼트에서 돌려줍니다 (만료 전까지)
tok_again = get_m2m_token()
print("같은 토큰 재사용?", tok_again == tok_m2m, "| jti:", jwt_claims(tok_again).get("jti"))

### 2-3. 보호된 API 쪽 검증
API는 토큰을 **서명(JWKS)·발급자·만료·스코프**로 검증해야 합니다. 스코프가 맞지 않으면 거부합니다. (`pyjwt[crypto]`가 없으면 클레임만 검사)

In [ ]:
JWKS_URL = f"{ISSUER}/.well-known/jwks.json"

def protected_api(token, need_scope):
    try:
        import jwt
        claims = jwt.decode(token, jwt.PyJWKClient(JWKS_URL).get_signing_key_from_jwt(token).key,
                            algorithms=["RS256"], issuer=ISSUER, options={"verify_aud": False})
        how = "서명검증"
    except ImportError:
        claims, how = jwt_claims(token), "클레임만"
        if claims["iss"] != ISSUER or claims["exp"] < time.time():
            return 401, how, "발급자/만료 오류"
    if need_scope not in claims.get("scope", "").split():
        return 403, how, f"스코프 부족 (보유: {claims.get('scope')})"
    who = claims.get("username") or f"app:{claims['client_id']}"
    return 200, how, f"OK — 호출자 {who}"

show_table([("read 요청 (M2M 토큰)", *protected_api(tok_m2m, API_SCOPE)),
            ("write 요청 (M2M 토큰)", *protected_api(tok_m2m, f"{API_ID}/write"))],
           ["요청", "HTTP", "검증", "결과"])

In [ ]:
# 서명 위조 실험 (pyjwt 필요): 페이로드의 scope를 조작하면 서명이 깨져 거부됩니다
h, p, s = tok_m2m.split(".")
forged_payload = jwt_claims(tok_m2m) | {"scope": f"{API_ID}/read {API_ID}/admin"}
forged = ".".join([h, base64.urlsafe_b64encode(json.dumps(forged_payload).encode()).decode().rstrip("="), s])
try:
    print(protected_api(forged, f"{API_ID}/admin"), "← pyjwt 미설치 시 클레임만 보므로 통과됨: 서명 검증이 왜 필수인지!")
except Exception as e:
    print("거부:", type(e).__name__, "—", e)

### 2-4. 같은 일을 SDK 데코레이터로
AgentCore Runtime에서 실제로 쓰는 형태입니다. Runtime은 워크로드 토큰을 컨텍스트에 넣어 주므로, 여기서는 그 동작을 `set_workload_access_token`으로 흉내 냅니다. 함수 본문은 **토큰을 인자로 받기만** 합니다.

In [ ]:
from bedrock_agentcore.identity.auth import requires_access_token
from bedrock_agentcore.runtime.context import BedrockAgentCoreContext

@requires_access_token(provider_name=M2M_PROVIDER, scopes=[API_SCOPE], auth_flow="M2M")
def report_agent_tool(*, access_token: str):
    return protected_api(access_token, API_SCOPE)

BedrockAgentCoreContext.set_workload_access_token(wat_m2m)   # Runtime이 해 주는 일
print(report_agent_tool())

## 3. 3LO — 사용자를 대신하는 권한 (USER_FEDERATION / authorization code)
```
사용자 ─▶ 에이전트 ──(워크로드 토큰 for user=student01)──▶ AgentCore Identity
                                   ◀── authorizationUrl + sessionUri ──┘
사용자 브라우저 ─▶ IdP 로그인·동의 ─▶ AgentCore 콜백(code→token 교환, 볼트 저장)
              ─▶ 우리 앱 return URL(?session_id=…) ─▶ CompleteResourceTokenAuth(session, user)
에이전트 ── 폴링 ──▶ access token (sub = 사용자)
```
마지막 `CompleteResourceTokenAuth`가 **세션 바인딩**입니다. 흐름을 *시작한 사용자*와 *로그인을 마친 사용자*가 같은지 앱이 확인해 주어야 토큰이 발급됩니다. 피싱으로 남에게 동의 링크를 눌리게 해도 공격자 세션에 토큰이 붙지 않습니다.

### 3-1. 자격 증명 공급자 등록 + IdP에 AgentCore 콜백 등록

In [ ]:
usr_cp = recreate(ctl.delete_oauth2_credential_provider, ctl.create_oauth2_credential_provider,
    name=USR_PROVIDER, credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput={"customOauth2ProviderConfig": {
        "oauthDiscovery": OAUTH_DISCOVERY,
        "clientId": usr_app["ClientId"], "clientSecret": client_secret(usr_app)}})

AGENTCORE_CALLBACK = usr_cp["callbackUrl"]
# update_user_pool_client는 생략한 필드를 기본값으로 되돌리므로 전체 설정을 다시 넘깁니다
cog.update_user_pool_client(ClientId=usr_app["ClientId"], CallbackURLs=[AGENTCORE_CALLBACK], **USR_CLIENT_KW)

print("[생성] 자격 증명 공급자 ", usr_cp["credentialProviderArn"])
print("       IdP에 등록한 콜백  :", AGENTCORE_CALLBACK)
print("       우리 앱 return URL :", RETURN_URL)

### 3-2. 우리 앱의 return URL 서버 (세션 바인딩 담당)
실서비스라면 웹 앱 백엔드가 **로그인된 사용자 세션**에서 사용자 ID를 꺼내 `CompleteResourceTokenAuth`를 부릅니다. 여기서는 `USER_ID`를 고정값으로 씁니다.

In [ ]:
USER_ID = "student-01"          # 우리 앱의 사용자 식별자 (IdP 계정명과 같을 필요 없음)
CALLBACK_LOG = []

def complete_session(session_uri, user_id):
    data.complete_resource_token_auth(sessionUri=session_uri, userIdentifier={"userId": user_id})

class ReturnHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        q = parse_qs(urlparse(self.path).query)
        sid = (q.get("session_id") or q.get("sessionUri") or [None])[0]
        if not sid:
            self.send_response(404); self.end_headers(); return
        try:
            complete_session(sid, USER_ID)
            msg = "인증 완료 — 노트북으로 돌아가세요."
            CALLBACK_LOG.append(("OK", sid[:40]))
        except ClientError as e:
            msg = "실패: " + e.response["Error"]["Message"]
            CALLBACK_LOG.append(("FAIL", msg))
        self.send_response(200); self.send_header("Content-Type", "text/html; charset=utf-8"); self.end_headers()
        self.wfile.write(f"<h2>{msg}</h2>".encode())
    def log_message(self, *a): pass

try:
    srv.shutdown(); srv.server_close()
except NameError:
    pass
srv = ThreadingHTTPServer(("127.0.0.1", RETURN_PORT), ReturnHandler)
threading.Thread(target=srv.serve_forever, daemon=True).start()
print("return URL 서버 대기:", RETURN_URL)

### 3-3. 사용자 토큰 요청 → 브라우저 로그인 → 토큰 수령
세 셀로 나뉩니다. (Jupyter는 한 번에 한 셀만 실행하므로, 기다리는 셀과 수동 완료 셀을 분리해야 서로 막지 않습니다.)

1. **(a) 로그인 링크 받기** — 링크를 눌러 0단계에 출력된 **테스트 사용자/비밀번호**로 로그인합니다.
2. **(b) 수동 완료 — 원격 커널(SageMaker 등)일 때만** — 로그인 후 브라우저가 `localhost:9090`을 열지 못하고 오류 페이지를 보여 줍니다. 이때 **주소창의 URL 전체**를 복사해 `redirected`에 붙여 넣고 실행합니다. 노트북을 로컬 PC에서 실행하면 콜백 서버가 이 일을 자동으로 하므로 건너뜁니다.
3. **(c) 토큰 기다리기** — 최대 2분 동안 토큰이 나올 때까지 확인합니다.

> **로그인 링크는 한 번만 쓸 수 있습니다.** 두 번 클릭하거나, 새로고침하거나, 로그인 페이지에서 뒤로 갔다 다시 열면 `{"message":"Invalid request"}`(HTTP 400)가 나옵니다. 몇 분 지나도 만료됩니다. 이럴 때는 **(a)를 다시 실행해 새 링크**를 받으세요. 이전 링크는 다시 쓰지 마세요.

In [ ]:
# (a) 로그인 링크 받기 — 이 셀은 기다리지 않고 바로 끝납니다
wat_user = data.get_workload_access_token_for_user_id(workloadName=WORKLOAD, userId=USER_ID)["workloadAccessToken"]

REQ_3LO = dict(workloadIdentityToken=wat_user, resourceCredentialProviderName=USR_PROVIDER,
               scopes=["openid", "email", "profile", API_SCOPE], oauth2Flow="USER_FEDERATION",
               resourceOauth2ReturnUrl=RETURN_URL)

r = data.get_resource_oauth2_token(forceAuthentication=True, **REQ_3LO)   # 데모용: 볼트에 있어도 다시 동의
SESSION_URI = r.get("sessionUri")
CALLBACK_LOG.clear()
print("sessionStatus:", r.get("sessionStatus"), "| sessionUri:", (SESSION_URI or "")[:60], "…")
display(HTML(f'<p style="font-size:1.2em">👉 <a href="{r["authorizationUrl"]}" target="_blank">여기를 눌러 로그인·동의</a>'
             f' &nbsp;(사용자 <code>{DEMO_USER}</code> / <code>{DEMO_PASS}</code>)</p>'))
print("※ 링크는 1회용 — 'Invalid request'가 보이면 이 셀을 다시 실행해 새 링크로 로그인")
print("로그인 후: 로컬 커널이면 (c)로, 브라우저에 localhost 연결 오류가 보이면 (b)로")

def poll_user_token(timeout=120):
    t0 = time.time()
    while time.time() - t0 < timeout:
        tok = data.get_resource_oauth2_token(sessionUri=SESSION_URI, **REQ_3LO).get("accessToken")
        if tok:
            return tok
        time.sleep(3)
    raise TimeoutError("시간 초과 — 로그인을 마쳤는지 확인하세요. 원격 커널이면 (b) 수동 완료를 먼저 실행해야 합니다.")

In [ ]:
# (b) 수동 완료 — 원격 커널일 때만. 로그인 후 브라우저 주소창의 localhost URL 전체를 붙여 넣고 실행
redirected = ""   # 예: "http://localhost:9090/oauth2/callback?session_id=urn%3Aietf%3Aparams%3Aoauth%3Arequest_uri%3A..."

if redirected:
    q = parse_qs(urlparse(redirected.strip()).query)
    sid = (q.get("session_id") or q.get("sessionUri") or [None])[0]
    if sid != SESSION_URI:
        print("[주의] (a)에서 받은 최신 링크의 세션이 아닙니다. (a)를 다시 실행해 새 링크로 로그인하세요.")
    else:
        complete_session(sid, USER_ID)
        CALLBACK_LOG.append(("MANUAL", sid[:40]))
        print("[완료] 세션 바인딩 — 이제 (c)를 실행하세요.")
else:
    print("건너뜀 (로컬 커널은 콜백 서버가 자동 처리)")

In [ ]:
# (c) 토큰 기다리기
tok_user = poll_user_token()
print("콜백 로그:", CALLBACK_LOG)
print("[수령] 사용자 위임 토큰:", short(tok_user))

### 3-4. 2LO 토큰 vs 3LO 토큰

In [ ]:
cm, cu = jwt_claims(tok_m2m), jwt_claims(tok_user)
show_table([(k, cm.get(k, "(없음)"), cu.get(k, "(없음)")) for k in ("sub", "client_id", "username", "scope", "token_use")],
           ["클레임", "2LO (M2M)", "3LO (사용자 위임)"])
print()
show_table([("2LO 토큰", *protected_api(tok_m2m, API_SCOPE)), ("3LO 토큰", *protected_api(tok_user, API_SCOPE))],
           ["토큰", "HTTP", "검증", "결과"])

### 3-5. 토큰 볼트 — 재동의 없이 재사용, 사용자별 격리
- 같은 사용자(`student-01`)가 다시 요청 → **브라우저 없이** 바로 토큰 (SDK 데코레이터 사용)
- 다른 사용자(`student-02`)가 같은 에이전트로 요청 → 토큰 대신 **로그인 URL** (남의 토큰을 받을 수 없음)

In [ ]:
@requires_access_token(provider_name=USR_PROVIDER, scopes=REQ_3LO["scopes"], auth_flow="USER_FEDERATION",
                       callback_url=RETURN_URL, on_auth_url=lambda url: print("로그인 필요:", url[:80]))
def my_data_tool(*, access_token: str):
    return jwt_claims(access_token)["username"], protected_api(access_token, API_SCOPE)[2]

BedrockAgentCoreContext.set_workload_access_token(wat_user)   # Runtime이 요청자 기준으로 넣어 주는 토큰
print("student-01 재요청:", my_data_tool())

wat_other = data.get_workload_access_token_for_user_id(workloadName=WORKLOAD, userId="student-02")["workloadAccessToken"]
r2 = data.get_resource_oauth2_token(**(REQ_3LO | {"workloadIdentityToken": wat_other}))
print("student-02 요청  :", "accessToken" if "accessToken" in r2 else "토큰 없음 → authorizationUrl 반환 (본인 동의 필요)")

### 3-6. (토론) 무엇이 막혔나
| 위협 | 2LO/3LO + AgentCore Identity에서의 방어 |
|---|---|
| 프롬프트 인젝션으로 “API 키를 출력해” | 에이전트 메모리·프롬프트에 secret이 **애초에 없음** (Secrets Manager) |
| 탈취된 토큰의 과도한 권한 | 스코프 최소화 (`read`만) → `write`·`admin`은 API가 403 |
| 토큰 위·변조 | API가 JWKS 서명 검증 |
| 다른 사용자 데이터 접근 | 볼트가 (워크로드, 사용자) 단위로 토큰 분리 |
| 동의 링크 피싱 / 리다이렉트 조작 | return URL 허용 목록 + `CompleteResourceTokenAuth` 세션 바인딩 |
| 감사 | 모든 `GetResourceOauth2Token` 호출이 CloudTrail에 기록 |

**과제**: 2-2에서 `scopes=[f"{API_ID}/write"]`로 요청하면 어디서 실패하나요? (IdP? 볼트? API?) / 3-2 서버의 `USER_ID`를 `"attacker"`로 바꾼 뒤 3-3을 다시 하면?

## 4. 정리 (반드시 실행)

In [ ]:
try:
    srv.shutdown(); srv.server_close(); print("[중지] return URL 서버")
except NameError:
    pass
for name in (M2M_PROVIDER, USR_PROVIDER):
    try:
        ctl.delete_oauth2_credential_provider(name=name); print("[삭제] 자격 증명 공급자", name)
    except ClientError as e:
        print("  건너뜀", name, e.response["Error"]["Code"])
try:
    ctl.delete_workload_identity(name=WORKLOAD); print("[삭제] 워크로드 아이덴티티", WORKLOAD)
except ClientError as e:
    print("  건너뜀", WORKLOAD, e.response["Error"]["Code"])
cleanup_cognito()